In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("exam-prep").master("local[*]").getOrCreate()
print(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/25 21:22:34 WARN Utils: Your hostname, Titass-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.27 instead (on interface en0)
26/11/25 21:22:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/homebrew/Caskroom/miniforge/base/envs/railweather/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/25 21:22:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


4.3.0


In [5]:
from pyspark.sql import functions as F, Window

In [6]:


spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")  # parses dates like "09-NOV-25"

WEATHER_CODES = ["X1","X2","X4","X9","XJ","XT","XW","QH","QI","QJ",
                 "IW","IP","JT","JH","JK","J6","OG"]

# Delays -> weather delay minutes per day
delays = (spark.read.option("header", True).option("inferSchema", True)
          .csv("../data/raw/Delays/Transparency 25-26 P04.csv"))

daily = (delays.filter(F.col("INCIDENT_REASON").isin(WEATHER_CODES))
         .withColumn("date", F.to_date("ORIGIN_DEPARTURE_DATE", "dd-MMM-yy"))
         .groupBy("date").agg(F.sum("PFPI_MINUTES").alias("weather_min")))

# Weather -> add a 3-day rolling rain feature per city (on the full history, before joining)
w = Window.partitionBy("city").orderBy("date").rowsBetween(-2, 0)
weather = (spark.read.option("header", True).option("inferSchema", True)
           .csv("../data/raw/weather_daily.csv")
           .withColumn("date", F.to_date("time"))
           .withColumn("rain_3d", F.sum("precipitation_sum").over(w)))

data = weather.join(daily, "date")   # keeps only the P04 dates
print(data.count(), "rows")          # expect ~28 days x 8 cities = 224

224 rows


In [7]:
cutoff = "2025-07-13"
train = data.filter(F.col("date") < cutoff)    # ~21 days
test  = data.filter(F.col("date") >= cutoff)   # last ~7 days

# Threshold from TRAINING days only (one value per day, hence distinct)
threshold = (train.select("date", "weather_min").distinct()
             .approxQuantile("weather_min", [0.75], 0)[0])
print("High-disruption threshold (minutes):", threshold)

add_label = lambda df: df.withColumn("label", (F.col("weather_min") >= threshold).cast("double"))
train, test = add_label(train), add_label(test)

High-disruption threshold (minutes): 2403.0


In [8]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression

features = ["temperature_2m_max", "temperature_2m_min", "precipitation_sum",
            "rain_3d", "wind_gusts_10m_max"]

indexer   = StringIndexer(inputCol="city", outputCol="city_idx", handleInvalid="keep")
encoder   = OneHotEncoder(inputCols=["city_idx"], outputCols=["city_ohe"])
assembler = VectorAssembler(inputCols=features + ["city_ohe"], outputCol="features")
lr        = LogisticRegression(featuresCol="features", labelCol="label")

pipeline = Pipeline(stages=[indexer, encoder, assembler, lr])
model = pipeline.fit(train)        # every estimator learns from TRAIN only
preds = model.transform(test)      # the fitted PipelineModel just applies them

26/11/25 21:25:02 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [9]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

ev = BinaryClassificationEvaluator(labelCol="label")      # default metric = areaUnderROC
print("areaUnderROC:", round(ev.evaluate(preds), 3))
print("areaUnderPR: ", round(ev.evaluate(preds, {ev.metricName: "areaUnderPR"}), 3))

(preds.withColumn("p_disrupt", vector_to_array("probability")[1])
      .groupBy("date")
      .agg(F.first("label").alias("actual"), F.round(F.avg("p_disrupt"), 2).alias("avg_predicted_p"))
      .orderBy("date").show())

areaUnderROC: 0.445
areaUnderPR:  0.128
+----------+------+---------------+
|      date|actual|avg_predicted_p|
+----------+------+---------------+
|2025-07-13|   0.0|            0.7|
|2025-07-14|   0.0|           0.39|
|2025-07-15|   1.0|           0.43|
|2025-07-16|   0.0|           0.03|
|2025-07-17|   0.0|           0.32|
|2025-07-18|   0.0|           0.78|
|2025-07-19|   0.0|           0.72|
+----------+------+---------------+



26/11/25 22:17:16 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 603619 ms exceeds timeout 120000 ms
26/11/25 22:17:16 WARN SparkContext: Killing executors is not supported by current scheduler.
26/11/25 22:17:21 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:85)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:841